# Paper 2 ECE v1.1: missingness-aware routing evidence
This notebook summarizes the Washington-calibrated routing policies evaluated on the five ECE collaboration in-situ stations in `derived_8.4_ece_v3`. ECE observations and targets are evaluation-only; Guarded's auxiliary class mapping is selected using Washington validation only.


## 1. Locate the evidence bundle
The next cell resolves the repository and canonical ECE v3 split for reproducible outputs.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (parent / "data" / "splits").is_dir() and (parent / "notebooks").is_dir()
)
EVIDENCE_DIR = PROJECT_ROOT / "notebooks/experiment/paper2-final-evidence-1.1"
ECE_DIR = EVIDENCE_DIR / "ece_guarded"
FIGURES_DIR = EVIDENCE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Evidence directory: {EVIDENCE_DIR}")


Evidence directory: /scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/paper2-final-evidence-1.1


## 2. Check the unseen ECE v3 evaluation set and site descriptors
The canonical test split contains 150 daily observations across five stations not used for Washington fitting. Site descriptors document the deployment context without implying that geography caused any routing outcome.


In [2]:
ECE_SPLIT = PROJECT_ROOT / "data/splits/derived_8.4_ece_v3"
ece_test = pd.read_csv(ECE_SPLIT / "test.csv", low_memory=False)
site_features = pd.read_csv(ECE_SPLIT / "station_static_features.csv")
assert len(ece_test) == 150
assert ece_test["station_id"].nunique() == 5
assert bool((ece_test["station_id"].value_counts() == 30).all())
site_columns = ["station_id", "J_elev_m", "J_bio_bio12"]
site_descriptors = site_features[site_columns].sort_values("station_id")
print(site_descriptors.to_string(index=False))

             station_id  J_elev_m  J_bio_bio12
    ECE_BBG_Lost_Meadow        52         1019
        ECE_BBG_Main_St        51         1018
ECE_Renton_Garden_North       157         1227
 ECE_Renton_Garden_Shed       157         1227
        ECE_Renton_Home       136         1181


## 3. Validate the ECE v3 run and export T7/T9 tables
The next cell checks five-seed family and policy coverage, confirms disjoint WA/ECE stations and full-SMAP-missing gating, audits the family-local oracle labels separately from the G_API mapping, and writes pooled and station-level tables.


In [3]:
summary = pd.read_csv(ECE_DIR / "summary.csv")
seed_metrics = pd.read_csv(ECE_DIR / "seed_metrics.csv")
station_metrics = pd.read_csv(ECE_DIR / "station_metrics.csv")
predictions = pd.read_csv(ECE_DIR / "predictions_v3.csv")
wa_calibration = pd.read_csv(ECE_DIR / "wa_calibration.csv")
with (ECE_DIR / "routing_audit.json").open(encoding="utf-8") as handle:
    routing_audit = json.load(handle)

wa_dir = PROJECT_ROOT / "data/splits/derived_8.4"
wa_stations = set(pd.read_csv(wa_dir / "train.csv", usecols=["station_id"])["station_id"])
wa_stations.update(pd.read_csv(wa_dir / "val.csv", usecols=["station_id"])["station_id"])

ece_split = PROJECT_ROOT / "data/splits/derived_8.4_ece_v3"
ece_test = pd.read_csv(ece_split / "test.csv", low_memory=False)
assert len(ece_test) == 150
assert ece_test["station_id"].nunique() == 5
assert bool((ece_test["station_id"].value_counts() == 30).all())
assert wa_stations.isdisjoint(set(ece_test["station_id"]))

families = ["Clustering_V0_Full_k2", "Clustering_Backbone54_k2", "Guarded_Backbone54_k2"]
policies = ["as_routed", "auto_hard", "auto_soft", "c0_only", "c1_only"]
seeds = [42, 7, 13, 101, 123]
assert set(seed_metrics["family"]) == set(families + ["Global_Single_54"])
assert set(seed_metrics["policy"]) == set(policies + ["direct"])
assert len(seed_metrics) == 80
assert len(station_metrics) == 400
assert len(predictions) == 11250
assert bool((summary.loc[summary["policy"].isin(["c0_only", "c1_only"]), "deployable"] == False).all())
assert routing_audit["ece_rows_used_for_fitting"] is False
assert routing_audit["ece_rows_used_for_mapping_or_calibration"] is False
expert_map = routing_audit["semantic_expert_indices"]
aux_map = routing_audit["aux_expert_mappings"]
assert expert_map["Clustering_V0_Full_k2"]["dry_expert_index"] == 0
assert expert_map["Clustering_Backbone54_k2"]["dry_expert_index"] == 0
assert expert_map["Guarded_Backbone54_k2"]["dry_expert_index"] == 1
assert aux_map["Clustering_V0_Full_k2"]["gapi_class_0_local_expert"] == 0
assert aux_map["Clustering_Backbone54_k2"]["gapi_class_0_local_expert"] == 0
guarded_map = aux_map["Guarded_Backbone54_k2"]
assert guarded_map["gapi_class_1_local_expert"] == 1 - guarded_map["gapi_class_0_local_expert"]
guard_candidates = wa_calibration.loc[
    wa_calibration["family"].eq("Guarded_Backbone54_k2")
    & wa_calibration["setting"].eq("smap_masked_val")
    & wa_calibration["policy"].str.startswith("aux_hard_candidate_")
]
assert len(guard_candidates) == 2
assert int(guard_candidates["selected_for_ece"].sum()) == 1
selected_guarded = guard_candidates.loc[guard_candidates["selected_for_ece"]].iloc[0]
assert int(selected_guarded["gapi_class_0_local_expert"]) == guarded_map["gapi_class_0_local_expert"]
assert np.isclose(selected_guarded["rmse"], guarded_map["selected_masked_wa_val_rmse"])

routing_audit_rows = []
for family in families:
    gate = routing_audit["ece_gate_audit"][family]
    assert gate["n_rows"] == 150 and gate["gate_rows"] == 150 and gate["gate_share"] == 1.0
    assert gate["mean_smap_miss_rate"] == 1.0
    semantic = expert_map[family]
    auxiliary = aux_map[family]
    for policy in policies:
        rows = seed_metrics.query("family == @family and policy == @policy")
        assert sorted(rows["seed"].astype(int).tolist()) == sorted(seeds)
    routed = predictions.query("family == @family and policy == 'auto_hard'")
    expected = np.where(
        routed["gapi_class"].to_numpy() == 0,
        int(aux_map[family]["gapi_class_0_local_expert"]),
        int(aux_map[family]["gapi_class_1_local_expert"]),
    )
    assert bool((routed["gapi_route_local_expert"].to_numpy() == expected).all())
    assert bool((routed["w0"].to_numpy() == (expected == 0)).all())
    routing_audit_rows.append({
        "family": family,
        "ece_rows": int(gate["n_rows"]),
        "availability_gated_rows": int(gate["gate_rows"]),
        "gate_share": float(gate["gate_share"]),
        "mean_smap_miss_rate": float(gate["mean_smap_miss_rate"]),
        "dry_assigned_local_expert_index": int(semantic["dry_expert_index"]),
        "complementary_local_expert_index": int(semantic["wet_expert_index"]),
        "oracle_convention_source": semantic["mapping_source"],
        "wa_canonical_feature_drier_index": int(semantic["wa_canonical_feature_drier_index"]),
        "dry_assignment_matches_wa_feature_minimum": bool(
            semantic["semantic_dry_matches_canonical_feature_minimum"]),
        "gapi_class_0_local_expert": int(auxiliary["gapi_class_0_local_expert"]),
        "gapi_class_1_local_expert": int(auxiliary["gapi_class_1_local_expert"]),
        "gapi_mapping_source": auxiliary["selection_source"],
        "selected_masked_wa_val_rmse": float(auxiliary["selected_masked_wa_val_rmse"]),
        "ece_used_for_fit_or_mapping": False,
    })
routing_audit_table = pd.DataFrame(routing_audit_rows)
routing_audit_table.to_csv(ECE_DIR / "routing_audit.csv", index=False)
print(f"WA stations={len(wa_stations)}; ECE stations={ece_test['station_id'].nunique()} (disjoint)")
print(f"seed_metrics rows={len(seed_metrics)} station_metrics rows={len(station_metrics)} predictions rows={len(predictions)}")
print(f"routing_audit.csv rows={len(routing_audit_table)}")
print(summary[["family", "policy", "deployable", "rmse_mean", "rmse_std", "bias_mean", "ubrmse_mean"]].to_string(index=False))
print("Approved dry-assignment convention (legacy salvage C0 for V0/Backbone; Guarded canonical c1):")
print(json.dumps(expert_map, indent=2))
print("WA-calibrated G_API class-to-local-expert mappings (separate from the oracle label convention):")
print(json.dumps(aux_map, indent=2))


WA stations=7; ECE stations=5 (disjoint)
seed_metrics rows=80 station_metrics rows=400 predictions rows=11250
routing_audit.csv rows=3
                  family    policy  deployable  rmse_mean  rmse_std  bias_mean  ubrmse_mean
   Clustering_V0_Full_k2 as_routed        True   0.165908  0.003051   0.117727     0.116899
   Clustering_V0_Full_k2 auto_hard        True   0.057768  0.000617   0.028654     0.050158
   Clustering_V0_Full_k2 auto_soft        True   0.057768  0.000617   0.028655     0.050158
   Clustering_V0_Full_k2   c0_only       False   0.057768  0.000617   0.028654     0.050158
   Clustering_V0_Full_k2   c1_only       False   0.192536  0.004021   0.185650     0.051020
Clustering_Backbone54_k2 as_routed        True   0.167431  0.003401   0.141932     0.088816
Clustering_Backbone54_k2 auto_hard        True   0.057768  0.000617   0.028654     0.050158
Clustering_Backbone54_k2 auto_soft        True   0.057768  0.000617   0.028655     0.050158
Clustering_Backbone54_k2   c0_only   

### Verify the direct Global reference coverage
The direct global reference has no local-expert routing policies, so this check verifies its five seeds and deployability separately from the routed-family loops.

In [4]:
global_direct = seed_metrics.query(
    "family == 'Global_Single_54' and policy == 'direct'"
)
assert sorted(global_direct["seed"].astype(int).tolist()) == sorted(seeds)
assert len(global_direct) == 5
global_summary = summary.query(
    "family == 'Global_Single_54' and policy == 'direct'"
)
assert len(global_summary) == 1
assert bool(global_summary["deployable"].iloc[0])
routed_summary = summary.query("policy in ['as_routed', 'auto_hard', 'auto_soft']")
assert bool(routed_summary["deployable"].all())
print(f"Global_Single_54 direct rows={len(global_direct)}; seeds={sorted(global_direct['seed'].astype(int).tolist())}; deployable=true")

Global_Single_54 direct rows=5; seeds=[7, 13, 42, 101, 123]; deployable=true


### Export the CSV tables used by the figures
This cell writes pooled, station-level, and site-descriptor summaries from the recorded run CSVs before the plotting cells read them.

In [5]:
site_features = pd.read_csv(ece_split / "station_static_features.csv")
site_columns = ["station_id", "J_elev_m", "J_bio_bio12"]
site_descriptors = site_features[site_columns].sort_values("station_id")
site_descriptors.to_csv(ECE_DIR / "site_descriptors.csv", index=False)
summary.to_csv(ECE_DIR / "ece_policy_summary.csv", index=False)
station_summary = station_metrics.groupby(
    ["family", "policy", "station"], as_index=False
).agg(
    rmse_mean=("rmse", "mean"),
    rmse_std=("rmse", "std"),
    bias_mean=("bias", "mean"),
    ubrmse_mean=("ubrmse", "mean"),
)
station_summary.to_csv(ECE_DIR / "ece_station_policy_summary.csv", index=False)
print("Pooled policy table:")
print(summary[["family", "policy", "deployable", "rmse_mean", "rmse_std",
               "bias_mean", "ubrmse_mean"]].to_string(index=False))
print(f"Station summary rows={len(station_summary)}; site descriptors={len(site_descriptors)}")

Pooled policy table:
                  family    policy  deployable  rmse_mean  rmse_std  bias_mean  ubrmse_mean
   Clustering_V0_Full_k2 as_routed        True   0.165908  0.003051   0.117727     0.116899
   Clustering_V0_Full_k2 auto_hard        True   0.057768  0.000617   0.028654     0.050158
   Clustering_V0_Full_k2 auto_soft        True   0.057768  0.000617   0.028655     0.050158
   Clustering_V0_Full_k2   c0_only       False   0.057768  0.000617   0.028654     0.050158
   Clustering_V0_Full_k2   c1_only       False   0.192536  0.004021   0.185650     0.051020
Clustering_Backbone54_k2 as_routed        True   0.167431  0.003401   0.141932     0.088816
Clustering_Backbone54_k2 auto_hard        True   0.057768  0.000617   0.028654     0.050158
Clustering_Backbone54_k2 auto_soft        True   0.057768  0.000617   0.028655     0.050158
Clustering_Backbone54_k2   c0_only       False   0.057768  0.000617   0.028654     0.050158
Clustering_Backbone54_k2   c1_only       False   0.192536  

### Washington expert means and machine-readable routing audit
This table keeps the approved family-specific dry-assigned oracle convention visible beside each family's Washington canonical-feature and target means and its separately selected G_API mapping.

In [6]:
for row in routing_audit_rows:
    family = row["family"]
    semantic = expert_map[family]
    for local_index in (0, 1):
        row[f"wa_canonical_smap_mean_local_c{local_index}"] = float(
            semantic["wa_canonical_feature_mean_by_expert"][str(local_index)])
        row[f"wa_target_mean_local_c{local_index}"] = float(
            semantic["wa_target_mean_by_expert"][str(local_index)])
    row["wa_canonical_feature_for_ordering"] = "SMAP_sm_pm_interp_rollmean30"
routing_audit_table = pd.DataFrame(routing_audit_rows)
routing_audit_table.to_csv(ECE_DIR / "routing_audit.csv", index=False)
print("WA expert means and routing audit (family-local indices):")
print(routing_audit_table.to_string(index=False, float_format=lambda value: f"{value:.6f}"))

WA expert means and routing audit (family-local indices):
                  family  ece_rows  availability_gated_rows  gate_share  mean_smap_miss_rate  dry_assigned_local_expert_index  complementary_local_expert_index                                                          oracle_convention_source  wa_canonical_feature_drier_index  dry_assignment_matches_wa_feature_minimum  gapi_class_0_local_expert  gapi_class_1_local_expert                                                          gapi_mapping_source  selected_masked_wa_val_rmse  ece_used_for_fit_or_mapping  wa_canonical_smap_mean_local_c0  wa_target_mean_local_c0  wa_canonical_smap_mean_local_c1  wa_target_mean_local_c1 wa_canonical_feature_for_ordering
   Clustering_V0_Full_k2       150                      150    1.000000             1.000000                                0                                 1                              derived_8.4-ece-router-salvage-2.0 c0=dry convention                                 1         

### Washington synthetic-SMAP-mask calibration
This cell reports static-versus-auxiliary validation results and both Guarded G_API class-to-expert candidates. Guarded's mapping is selected from these Washington validation scores before evaluating the held-out ECE station set.


In [7]:
base_masked = wa_calibration.query(
    "setting == 'smap_masked_val' and policy in ['static_hard_masked', 'aux_hard_masked']"
)
assert len(base_masked) == 6
assert set(base_masked["family"]) == set(families)
guarded_candidates = wa_calibration.loc[
    wa_calibration["family"].eq("Guarded_Backbone54_k2")
    & wa_calibration["setting"].eq("smap_masked_val")
    & wa_calibration["policy"].str.startswith("aux_hard_candidate_")
]
assert len(guarded_candidates) == 2
assert int(guarded_candidates["selected_for_ece"].sum()) == 1
print("Standard static / selected-auxiliary synthetic-mask results:")
print(base_masked.sort_values(["family", "policy"]).to_string(index=False))
print("Guarded candidate mappings; selected mapping is frozen before ECE evaluation:")
print(guarded_candidates.sort_values("gapi_class_0_local_expert").to_string(index=False))


Standard static / selected-auxiliary synthetic-mask results:
                  family         setting             policy     rmse  gapi_class_0_local_expert  gapi_class_1_local_expert selected_for_ece
Clustering_Backbone54_k2 smap_masked_val    aux_hard_masked 0.073965                        NaN                        NaN              NaN
Clustering_Backbone54_k2 smap_masked_val static_hard_masked 0.086870                        NaN                        NaN              NaN
   Clustering_V0_Full_k2 smap_masked_val    aux_hard_masked 0.072757                        NaN                        NaN              NaN
   Clustering_V0_Full_k2 smap_masked_val static_hard_masked 0.068082                        NaN                        NaN              NaN
   Guarded_Backbone54_k2 smap_masked_val    aux_hard_masked 0.073965                        NaN                        NaN              NaN
   Guarded_Backbone54_k2 smap_masked_val static_hard_masked 0.086870                        NaN    

## 4. Export routing shares and calibration evidence
The next cell summarizes weights on the family-convention dry-assigned expert and its complement, then prints the Washington-only calibration and mapping-selection rows.


In [8]:
predictions["dry_assigned_weight"] = np.where(
    predictions["dry_expert_index"].astype(int) == 0,
    predictions["w0"], predictions["w1"])
predictions["complementary_weight"] = 1.0 - predictions["dry_assigned_weight"]
share_keys = ["family", "policy"]
routing_shares = predictions.groupby(share_keys, as_index=False).agg(
    dry_assigned_weight_mean=("dry_assigned_weight", "mean"),
    complementary_weight_mean=("complementary_weight", "mean"),
    dry_assigned_dominant_fraction=(
        "dry_assigned_weight", lambda values: float((values > 0.5).mean())))
routing_shares.to_csv(ECE_DIR / "ece_routing_shares.csv", index=False)
station_shares = predictions.groupby(["family", "policy", "station_id"], as_index=False).agg(
    dry_assigned_weight_mean=("dry_assigned_weight", "mean"),
    complementary_weight_mean=("complementary_weight", "mean"))
station_shares.to_csv(ECE_DIR / "ece_routing_shares_by_station.csv", index=False)
print(routing_shares.to_string(index=False))
print("WA-only calibration and Guarded mapping-selection rows:")
print(wa_calibration.to_string(index=False))


                  family    policy  dry_assigned_weight_mean  complementary_weight_mean  dry_assigned_dominant_fraction
Clustering_Backbone54_k2 as_routed              3.066667e-01               6.933333e-01                        0.306667
Clustering_Backbone54_k2 auto_hard              1.000000e+00               0.000000e+00                        1.000000
Clustering_Backbone54_k2 auto_soft              9.999995e-01               4.564617e-07                        1.000000
Clustering_Backbone54_k2   c0_only              1.000000e+00               0.000000e+00                        1.000000
Clustering_Backbone54_k2   c1_only              0.000000e+00               1.000000e+00                        0.000000
   Clustering_V0_Full_k2 as_routed              4.600000e-01               5.400000e-01                        0.460000
   Clustering_V0_Full_k2 auto_hard              1.000000e+00               0.000000e+00                        1.000000
   Clustering_V0_Full_k2 auto_soft      

## 4a. Export the family-local policy crosswalk
This crosswalk keeps oracle label conventions separate from the G_API class-to-expert mapping, records each local index and its Washington-only source, and marks oracle/diagnostic rows non-deployable.


In [9]:
policy_crosswalk_rows = []
for family, family_map in expert_map.items():
    aux_family_map = aux_map[family]
    for policy in policies:
        if policy == "c0_only":
            semantic_label = "manual dry-assigned oracle under family convention"
            expert_index = int(family_map["dry_expert_index"])
        elif policy == "c1_only":
            semantic_label = "complementary local-expert diagnostic under family convention"
            expert_index = int(family_map["wet_expert_index"])
        else:
            semantic_label = {
                "as_routed": "static router prediction",
                "auto_hard": "WA-calibrated availability-gated G_API route",
                "auto_soft": "WA-calibrated availability-gated G_API blend",
            }[policy]
            expert_index = None
        policy_crosswalk_rows.append({
            "family": family,
            "policy_id": policy,
            "semantic_label": semantic_label,
            "family_local_expert_index": expert_index,
            "dry_assigned_local_expert_index": int(family_map["dry_expert_index"]),
            "complementary_local_expert_index": int(family_map["wet_expert_index"]),
            "oracle_label_source": family_map["mapping_source"],
            "wa_canonical_feature_for_ordering": "SMAP_sm_pm_interp_rollmean30",
            "wa_canonical_smap_mean_local_c0": float(
                family_map["wa_canonical_feature_mean_by_expert"]["0"]),
            "wa_canonical_smap_mean_local_c1": float(
                family_map["wa_canonical_feature_mean_by_expert"]["1"]),
            "wa_target_mean_local_c0": float(
                family_map["wa_target_mean_by_expert"]["0"]),
            "wa_target_mean_local_c1": float(
                family_map["wa_target_mean_by_expert"]["1"]),
            "wa_canonical_feature_drier_index": int(
                family_map["wa_canonical_feature_drier_index"]),
            "dry_assignment_matches_canonical_feature_minimum": bool(
                family_map["semantic_dry_matches_canonical_feature_minimum"]),
            "gapi_class_0_local_expert": int(
                aux_family_map["gapi_class_0_local_expert"]),
            "gapi_class_1_local_expert": int(
                aux_family_map["gapi_class_1_local_expert"]),
            "gapi_mapping_source": aux_family_map["selection_source"],
            "gapi_selected_masked_wa_val_rmse": float(
                aux_family_map["selected_masked_wa_val_rmse"]),
            "deployable": policy in {"as_routed", "auto_hard", "auto_soft"},
            "ece_used_for_mapping_or_calibration": False,
        })
policy_crosswalk_rows.append({
    "family": "Global_Single_54",
    "policy_id": "direct",
    "semantic_label": "direct global reference",
    "family_local_expert_index": None,
    "dry_assigned_local_expert_index": None,
    "complementary_local_expert_index": None,
    "oracle_label_source": "not_applicable",
    "wa_canonical_feature_for_ordering": "not_applicable",
    "wa_canonical_smap_mean_local_c0": None,
    "wa_canonical_smap_mean_local_c1": None,
    "wa_target_mean_local_c0": None,
    "wa_target_mean_local_c1": None,
    "wa_canonical_feature_drier_index": None,
    "dry_assignment_matches_canonical_feature_minimum": None,
    "gapi_class_0_local_expert": None,
    "gapi_class_1_local_expert": None,
    "gapi_mapping_source": "not_applicable",
    "gapi_selected_masked_wa_val_rmse": None,
    "deployable": True,
    "ece_used_for_mapping_or_calibration": False,
})
policy_crosswalk = pd.DataFrame(policy_crosswalk_rows)
policy_crosswalk.to_csv(ECE_DIR / "policy_crosswalk.csv", index=False)
print(policy_crosswalk.to_string(index=False))


                  family policy_id                                                semantic_label  family_local_expert_index  dry_assigned_local_expert_index  complementary_local_expert_index                                                               oracle_label_source wa_canonical_feature_for_ordering  wa_canonical_smap_mean_local_c0  wa_canonical_smap_mean_local_c1  wa_target_mean_local_c0  wa_target_mean_local_c1  wa_canonical_feature_drier_index dry_assignment_matches_canonical_feature_minimum  gapi_class_0_local_expert  gapi_class_1_local_expert                                                          gapi_mapping_source  gapi_selected_masked_wa_val_rmse  deployable  ece_used_for_mapping_or_calibration
   Clustering_V0_Full_k2 as_routed                                      static router prediction                        NaN                              0.0                               1.0                              derived_8.4-ece-router-salvage-2.0 c0=dry convention      SM

## 5. Plot T7 and T9 ECE v3 diagnostics
The following figure uses the exported policy, station, routing-share, and site-descriptor CSVs; labels distinguish deployable policies from non-deployable diagnostics.


In [10]:
ece_policy = pd.read_csv(ECE_DIR / "ece_policy_summary.csv")
ece_station = pd.read_csv(ECE_DIR / "ece_station_policy_summary.csv")
route_station = pd.read_csv(ECE_DIR / "ece_routing_shares_by_station.csv")
site_descriptors = pd.read_csv(ECE_DIR / "site_descriptors.csv")

display_policy = {
    "as_routed": "Static route",
    "auto_hard": "Availability gate + G_API",
    "auto_soft": "Gate + G_API blend",
    "c0_only": "Dry-assigned oracle*",
    "c1_only": "Complementary diagnostic*",
    "direct": "Global direct",
}
family_order = [
    "Clustering_V0_Full_k2", "Clustering_Backbone54_k2",
    "Guarded_Backbone54_k2", "Global_Single_54",
]
family_label = {
    "Clustering_V0_Full_k2": "V0",
    "Clustering_Backbone54_k2": "Backbone",
    "Guarded_Backbone54_k2": "Guarded",
    "Global_Single_54": "Global",
}
policy_order = ["as_routed", "auto_hard", "auto_soft", "c0_only", "c1_only", "direct"]
policy_colors = dict(zip(policy_order, plt.cm.tab10.colors[:len(policy_order)]))

fig, axes = plt.subplots(2, 2, figsize=(17, 11))
ax = axes[0, 0]
x = np.arange(len(family_order))
bar_width = 0.12
for j, policy in enumerate(policy_order):
    rows = ece_policy[ece_policy["policy"] == policy].set_index("family")
    positions = x + (j - (len(policy_order) - 1) / 2) * bar_width
    means, errors = [], []
    for family in family_order:
        if family in rows.index:
            means.append(float(rows.loc[family, "rmse_mean"]))
            errors.append(float(rows.loc[family, "rmse_std"]))
        else:
            means.append(np.nan)
            errors.append(0.0)
    ax.bar(positions, means, bar_width, yerr=errors, capsize=2,
           color=policy_colors[policy], label=display_policy[policy])
ax.set_xticks(x, [family_label[family] for family in family_order])
ax.set_ylabel("ECE v3 RMSE (soil-moisture units)")
ax.set_title("T7 · pooled RMSE by policy (mean ± seed SD)")
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8, ncol=2)

ax = axes[0, 1]
guarded_station = ece_station[ece_station["family"] == "Guarded_Backbone54_k2"]
station_order = sorted(guarded_station["station"].unique())
x = np.arange(len(station_order))
bar_width = 0.14
for j, policy in enumerate(policy_order[:-1]):
    rows = guarded_station[guarded_station["policy"] == policy].set_index("station")
    positions = x + (j - 2) * bar_width
    means = [float(rows.loc[station, "rmse_mean"]) for station in station_order]
    errors = [float(rows.loc[station, "rmse_std"]) for station in station_order]
    ax.bar(positions, means, bar_width, yerr=errors, capsize=2,
           color=policy_colors[policy], label=display_policy[policy])
ax.set_xticks(x, [station.replace("ECE_", "").replace("_", " ")
                  for station in station_order], fontsize=8)
ax.set_ylabel("ECE v3 RMSE")
ax.set_title("T7 · Guarded station-level RMSE (mean ± seed SD)")
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8, ncol=2)

ax = axes[1, 0]
guarded_route = route_station[route_station["family"] == "Guarded_Backbone54_k2"]
descriptor_order = site_descriptors.set_index("station_id").loc[station_order]
for policy in ["as_routed", "auto_hard", "auto_soft"]:
    rows = guarded_route[guarded_route["policy"] == policy].set_index("station_id").loc[station_order]
    ax.plot(np.arange(len(station_order)), rows["dry_assigned_weight_mean"].to_numpy(),
            marker="o", linewidth=2, color=policy_colors[policy],
            label=display_policy[policy])
ax.set_xticks(np.arange(len(station_order)), [
    f"{station.replace('ECE_', '').replace('_', ' ')} ({int(elevation)}m)"
    for station, elevation in zip(station_order, descriptor_order["J_elev_m"])
], fontsize=8)
ax.set_ylim(-0.05, 1.05)
ax.set_ylabel("Mean weight on family-convention dry-assigned expert")
ax.set_title("T9 · Guarded routing share by site")
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8)

ax = axes[1, 1]
label_offsets = {
    "ECE_BBG_Main_St": (-64, -20),
    "ECE_BBG_Lost_Meadow": (7, 8),
    "ECE_Renton_Garden_North": (-116, 15),
    "ECE_Renton_Garden_Shed": (-114, -24),
    "ECE_Renton_Home": (7, 8),
}
for _, row in site_descriptors.iterrows():
    ax.scatter(row["J_elev_m"], row["J_bio_bio12"], color="#286c8e", s=48)
    short = row["station_id"].replace("ECE_", "").replace("_", " ")
    ax.annotate(short, (row["J_elev_m"], row["J_bio_bio12"]),
                xytext=label_offsets[row["station_id"]], textcoords="offset points",
                fontsize=8)
ax.set_xlabel("Elevation (m)")
ax.set_ylabel("Annual precipitation proxy (J_bio_bio12)")
ax.set_title("T9 · ECE station site descriptors")
ax.grid(alpha=0.25)

fig.suptitle("ECE v3 test diagnostics · five unseen stations · 30 daily observations each",
             fontsize=14)
fig.text(0.01, 0.01, "* Oracle and complementary diagnostic rows are non-deployable "
         "and use family-specific local indices (Guarded dry-assigned = c1; "
         "V0/Backbone = c0). These conventions are not physical classes. "
         "Guarded's G_API mapping was selected on WA validation.", fontsize=9)
fig.tight_layout(rect=(0, 0.04, 1, 0.96))
ece_t7_t9_path = FIGURES_DIR / "T7_T9_ece_diagnostics.png"
fig.savefig(ece_t7_t9_path, dpi=200, bbox_inches="tight")
plt.close(fig)

print(ece_policy[["family", "policy", "deployable", "rmse_mean", "rmse_std",
                  "bias_mean", "ubrmse_mean"]].to_string(index=False))
print(f"Saved figure: {ece_t7_t9_path}")


                  family    policy  deployable  rmse_mean  rmse_std  bias_mean  ubrmse_mean
   Clustering_V0_Full_k2 as_routed        True   0.165908  0.003051   0.117727     0.116899
   Clustering_V0_Full_k2 auto_hard        True   0.057768  0.000617   0.028654     0.050158
   Clustering_V0_Full_k2 auto_soft        True   0.057768  0.000617   0.028655     0.050158
   Clustering_V0_Full_k2   c0_only       False   0.057768  0.000617   0.028654     0.050158
   Clustering_V0_Full_k2   c1_only       False   0.192536  0.004021   0.185650     0.051020
Clustering_Backbone54_k2 as_routed        True   0.167431  0.003401   0.141932     0.088816
Clustering_Backbone54_k2 auto_hard        True   0.057768  0.000617   0.028654     0.050158
Clustering_Backbone54_k2 auto_soft        True   0.057768  0.000617   0.028655     0.050158
Clustering_Backbone54_k2   c0_only       False   0.057768  0.000617   0.028654     0.050158
Clustering_Backbone54_k2   c1_only       False   0.192536  0.004021   0.185650  

In [11]:
for chart_axis in axes.flat:
    chart_legend = chart_axis.get_legend()
    if chart_legend is not None:
        chart_legend.remove()
for annotation in axes[1, 1].texts:
    if annotation.get_text() == "BBG Main St":
        annotation.set_position((8, -18))
    elif annotation.get_text() == "Renton Garden North":
        annotation.set_position((-118, -28))
    elif annotation.get_text() == "Renton Garden Shed":
        annotation.set_position((8, -12))
axes[1, 1].set_xlim(35, 190)
axes[1, 1].set_ylim(990, 1250)
legend_handles, legend_labels = axes[0, 0].get_legend_handles_labels()
fig.legend(legend_handles, legend_labels, loc="upper center",
           bbox_to_anchor=(0.5, 0.945), ncol=3, fontsize=8, frameon=False)
fig.tight_layout(rect=(0, 0.04, 1, 0.87))
fig.savefig(ece_t7_t9_path, dpi=200, bbox_inches="tight")
print("Adjusted T7/T9 legends and site labels outside the data regions.")


Adjusted T7/T9 legends and site labels outside the data regions.


## 6. Plot the Guarded F5 prediction overlay
This section averages predictions across the five expert seeds by date and exports the daily table used by the five-station overlay. Oracle and complementary diagnostic curves are labeled non-deployable and remain family-index conventions.


In [12]:
guarded_family = "Guarded_Backbone54_k2"
overlay_policies = ["as_routed", "auto_hard", "c0_only", "c1_only"]
overlay = predictions[
    (predictions["family"] == guarded_family)
    & predictions["policy"].isin(overlay_policies)
].copy()
overlay["date"] = pd.to_datetime(overlay["date"])
assert overlay["seed"].nunique() == 5
assert set(overlay["station_id"]) == set(ece_test["station_id"])
assert set(overlay["policy"]) == set(overlay_policies)
overlay_daily = overlay.groupby(
    ["station_id", "date", "policy"], as_index=False
).agg(y_true=("y_true", "mean"), y_pred_mean=("y_pred", "mean"),
      y_pred_seed_sd=("y_pred", "std"))
target_checks = overlay.groupby(["station_id", "date"])["y_true"].nunique()
assert bool((target_checks == 1).all())
overlay_daily.to_csv(ECE_DIR / "f5_guarded_daily_overlay.csv", index=False)

display_overlay = {
    "as_routed": "Guarded · static route",
    "auto_hard": "Guarded · WA-calibrated availability gate + G_API",
    "c0_only": "Guarded · dry-assigned oracle (local c1)*",
    "c1_only": "Guarded · complementary diagnostic (local c0)*",
}
overlay_colors = {
    "as_routed": "#444444", "auto_hard": "#1b9e77",
    "c0_only": "#377eb8", "c1_only": "#d95f02",
}
fig, axes = plt.subplots(5, 1, figsize=(12, 13), sharex=True, sharey=True)
for ax, station in zip(axes, sorted(overlay_daily["station_id"].unique())):
    station_data = overlay_daily[overlay_daily["station_id"] == station]
    observed = station_data.groupby("date", as_index=False)["y_true"].mean()
    ax.plot(observed["date"], observed["y_true"], color="black", linewidth=1.5,
            label="Observed in-situ target")
    for policy in overlay_policies:
        rows = station_data[station_data["policy"] == policy].sort_values("date")
        ax.plot(rows["date"], rows["y_pred_mean"], color=overlay_colors[policy],
                linewidth=1.2, label=display_overlay[policy])
    ax.set_title(station.replace("ECE_", "").replace("_", " "), loc="left", fontsize=10)
    ax.grid(alpha=0.2)
    ax.set_ylabel("Soil moisture")
axes[-1].set_xlabel("Date")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, 0.035))
fig.suptitle("F5 · Guarded ECE v3 test observations (mean across five expert seeds)",
             y=0.995, fontsize=14)
fig.text(0.01, 0.005, "* Non-deployable family-convention rows; the approved Guarded "
         "dry-assigned oracle uses local c1 and the complementary diagnostic uses local c0.",
         fontsize=8)
fig.tight_layout(rect=(0, 0.10, 1, 0.98))
f5_path = FIGURES_DIR / "F5_guarded_ece_overlay.png"
fig.savefig(f5_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Daily table rows={len(overlay_daily)}")
print(f"Saved figure: {f5_path}")


Daily table rows=600
Saved figure: /scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/paper2-final-evidence-1.1/figures/F5_guarded_ece_overlay.png


### Make the overlapping Guarded routes distinguishable
The selected Guarded G_API map sends class 0 to local c0, while the approved family dry-assigned oracle is local c1. Since all ECE G_API predictions are class 0, auto_hard coincides with the complementary local c0 diagnostic; the next cell verifies this and uses a dashed line for the automatic policy.

In [13]:
guarded_auto = predictions.query(
    "family == 'Guarded_Backbone54_k2' and policy == 'auto_hard'"
)
guarded_complement = predictions.query(
    "family == 'Guarded_Backbone54_k2' and policy == 'c1_only'"
)
assert set(guarded_auto["gapi_class"].astype(int)) == {0}
matched = guarded_auto.merge(
    guarded_complement,
    on=["seed", "station_id", "date"],
    suffixes=("_auto", "_complement"),
    validate="one_to_one",
)
assert len(matched) == 750
assert bool(np.allclose(
    matched["y_pred_auto"].to_numpy(),
    matched["y_pred_complement"].to_numpy(),
    rtol=0.0,
    atol=1e-12,
))
for axis in axes:
    for line in axis.get_lines():
        label = line.get_label()
        if label == "Guarded · WA-calibrated availability gate + G_API":
            line.set_linestyle("--")
            line.set_linewidth(2.0)
            line.set_zorder(5)
        elif label == "Guarded · complementary diagnostic (local c0)*":
            line.set_zorder(3)
fig.savefig(f5_path, dpi=200, bbox_inches="tight")
print("Guarded auto_hard has G_API class 0 on all 750 prediction rows.")
print("Guarded auto_hard predictions match the local c0 complementary diagnostic; the automatic line is dashed.")

Guarded auto_hard has G_API class 0 on all 750 prediction rows.
Guarded auto_hard predictions match the local c0 complementary diagnostic; the automatic line is dashed.


## 7. Compare family-convention oracle rows with salvage C0
This seed-paired diagnostic preserves the approved V0/Backbone salvage C0 and Guarded c1 dry-assignment conventions. It compares non-deployable oracle rows only; it does not select the Guarded automatic mapping or use ECE outcomes for calibration.


In [14]:
salvage_seed = pd.read_csv(EVIDENCE_DIR / "comparators/ece_router_salvage_2.0_seed_metrics.csv")
comparison_specs = [
    ("V0 current vs salvage C0", "Clustering_V0_Full_k2", "Clustering_V0_Full_k2", 0),
    ("Backbone current vs salvage C0", "Clustering_Backbone54_k2", "Clustering_Backbone54_k2", 0),
    ("Guarded family-convention dry-assigned c1 vs salvage Backbone C0",
     "Guarded_Backbone54_k2", "Clustering_Backbone54_k2", 1),
]
paired_rows = []
for comparison_label, current_family, reference_family, current_dry_index in comparison_specs:
    current = seed_metrics.query(
        "family == @current_family and policy == 'c0_only'"
    )[["seed", "rmse", "deployable"]].rename(columns={
        "rmse": "current_rmse", "deployable": "current_deployable"
    })
    reference = salvage_seed.query(
        "family == @reference_family and policy == 'c0_only'"
    )[["seed", "rmse", "deployable"]].rename(columns={
        "rmse": "salvage_c0_rmse", "deployable": "salvage_deployable"
    })
    paired = current.merge(reference, on="seed", validate="one_to_one")
    assert sorted(paired["seed"].astype(int).tolist()) == sorted(seeds)
    paired["comparison"] = comparison_label
    paired["current_family"] = current_family
    paired["reference_family"] = reference_family
    paired["current_policy"] = "c0_only family-convention dry-assigned oracle"
    paired["reference_policy"] = "c0_only historical salvage C0 oracle"
    paired["current_local_dry_expert_index"] = current_dry_index
    paired["reference_local_dry_expert_index"] = 0
    paired["delta_rmse_current_minus_salvage"] = (
        paired["current_rmse"] - paired["salvage_c0_rmse"]
    )
    paired["deployable"] = False
    paired["ece_used_for_comparison_or_fit"] = False
    paired_rows.append(paired)

paired_dry_comparison = pd.concat(paired_rows, ignore_index=True)
assert len(paired_dry_comparison) == 15
assert bool((paired_dry_comparison["current_deployable"] == False).all())
assert bool((paired_dry_comparison["salvage_deployable"] == False).all())
paired_dry_comparison.to_csv(ECE_DIR / "salvage_c0_dry_comparison_by_seed.csv", index=False)
dry_comparison_summary = paired_dry_comparison.groupby("comparison", as_index=False).agg(
    current_rmse_mean=("current_rmse", "mean"),
    current_rmse_sd=("current_rmse", "std"),
    salvage_c0_rmse_mean=("salvage_c0_rmse", "mean"),
    salvage_c0_rmse_sd=("salvage_c0_rmse", "std"),
    delta_rmse_mean=("delta_rmse_current_minus_salvage", "mean"),
    delta_rmse_sd=("delta_rmse_current_minus_salvage", "std"),
    n_seeds=("seed", "nunique"))
dry_comparison_summary.to_csv(ECE_DIR / "salvage_c0_dry_comparison_summary.csv", index=False)
print(dry_comparison_summary.to_string(index=False, float_format=lambda value: f"{value:.6f}"))


                                                      comparison  current_rmse_mean  current_rmse_sd  salvage_c0_rmse_mean  salvage_c0_rmse_sd  delta_rmse_mean  delta_rmse_sd  n_seeds
                                  Backbone current vs salvage C0           0.057768         0.000617              0.057768            0.000617         0.000000       0.000000        5
Guarded family-convention dry-assigned c1 vs salvage Backbone C0           0.192536         0.004021              0.057768            0.000617         0.134768       0.003603        5
                                        V0 current vs salvage C0           0.057768         0.000617              0.057768            0.000617         0.000000       0.000000        5
